# 🎵 RagaTherapy: AI-Powered Disease-Aware Raga Recommendation System

### Complete Cell-by-Cell Colab Notebook

**Steps to run:**
1. Set runtime to GPU: **Runtime → Change runtime type → T4 GPU**
2. Run Cell 1 (installs + imports)
3. Run Cell 2 (upload the 3 `.py` files when prompted)
4. Run Cell 3 (executes everything automatically)
5. Check outputs in the Files panel

**Total runtime: ~15-25 minutes**

This notebook orchestrates the pipeline via three companion files (`ragatherapy_colab_cells.py`, `publication_enhancements.py`, `quality_fixes.py`) uploaded at runtime. For a standalone version that runs directly against the knowledge base in `src/data/` without those companion files, see `reproduction/reproduce_pipeline.py` at the repository root.

## Cell 1: Install All Libraries

In [ ]:
!pip install -q pandas numpy scikit-learn nltk sentence-transformers xgboost shap joblib matplotlib seaborn requests torch

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import seaborn as sns
import json, os, re, ast, random, requests, warnings
warnings.filterwarnings('ignore')

import nltk
for pkg in ['punkt', 'punkt_tab', 'stopwords', 'wordnet']:
    nltk.download(pkg, quiet=True)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix,
    top_k_accuracy_score, ndcg_score
)
from sklearn.dummy import DummyClassifier
from sentence_transformers import SentenceTransformer
import xgboost as xgb
import shap
import joblib

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ All libraries installed and imported')
print(f'   PyTorch {torch.__version__}, Device: {device}')
print(f'   CUDA: {torch.cuda.is_available()}')

## Cell 2: Upload Source Files

**Upload these 3 files when prompted:**
1. `ragatherapy_colab_cells.py`
2. `publication_enhancements.py`
3. `quality_fixes.py`

All 3 files are in the `notebooks/` folder of the project.

In [ ]:
# Upload the 3 source files
from google.colab import files

required_files = [
    'ragatherapy_colab_cells.py',
    'publication_enhancements.py',
    'quality_fixes.py'
]

missing = [f for f in required_files if not os.path.exists(f)]

if missing:
    print(f'📁 Please upload these files: {missing}')
    print('   (They are in the notebooks/ folder of the project)\n')
    uploaded = files.upload()
    print(f'\n✅ Uploaded {len(uploaded)} files')
else:
    print('✅ All source files already present')

# Verify
for f in required_files:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024
        print(f'   ✓ {f} ({size:.0f} KB)')
    else:
        print(f'   ✗ {f} — MISSING! Please re-upload.')

## Cell 3: Run Complete Pipeline

This runs ALL sections:
- Sections 2-18 from main notebook (datasets, KB, EDA, NLP, models, evaluation)
- Publication enhancements (K-fold, baselines, ablation)
- Quality fixes (evidence levels, SBERT fine-tuning, clustering)

**Expected runtime: 15-25 minutes**

In [ ]:
def run_py_file(filepath, skip_imports=True, skip_from_cell=None):
    """Read a .py file, strip shell commands, and execute it."""
    if not os.path.exists(filepath):
        print(f'⚠ {filepath} not found, skipping')
        return
    
    with open(filepath, 'r', encoding='utf-8') as f:
        code = f.read()
    
    lines = code.split('\n')
    clean = []
    started = (skip_from_cell is None)
    
    for line in lines:
        s = line.strip()
        
        # Skip until we reach the target cell
        if not started:
            if skip_from_cell and skip_from_cell in line:
                started = True
            continue
        
        # Skip shell commands
        if s.startswith('!pip') or s.startswith('!apt'):
            continue
        
        # Skip duplicate imports if requested
        if skip_imports:
            if s.startswith('import pandas as pd'): continue
            if s.startswith('import numpy as np'): continue
            if s.startswith('import matplotlib') and 'plt' not in s: continue
            if 'matplotlib.rcParams' in s: continue
            if s.startswith('import seaborn'): continue
            if s.startswith('import json') and ',' in s: continue
            if s.startswith('import torch') and 'nn' not in s and 'optim' not in s: continue
            if s.startswith('device = torch.device'): continue
        
        clean.append(line)
    
    exec_code = '\n'.join(clean)
    print(f'📄 Executing {filepath} ({len(clean)} lines)...')
    exec(exec_code, globals())
    print(f'✅ {filepath} complete\n')

# ---- RUN MAIN PIPELINE (Cells 3-18) ----
print('=' * 70)
print('PHASE 1: MAIN PIPELINE (Cells 3-18)')
print('=' * 70)
run_py_file('ragatherapy_colab_cells.py', skip_imports=True, skip_from_cell='CELL 3')

# ---- RUN PUBLICATION ENHANCEMENTS ----
print('=' * 70)
print('PHASE 2: PUBLICATION ENHANCEMENTS')
print('=' * 70)
run_py_file('publication_enhancements.py', skip_imports=True)

# ---- RUN QUALITY FIXES ----
print('=' * 70)
print('PHASE 3: QUALITY FIXES')
print('=' * 70)
run_py_file('quality_fixes.py', skip_imports=True)

print('\n' + '=' * 70)
print('🎉 ALL PHASES COMPLETE!')
print('=' * 70)

## Cell 4: Check All Outputs

In [ ]:
import glob

print('=' * 70)
print('📁 ALL GENERATED FILES')
print('=' * 70)

# Figures
figs = sorted(glob.glob('reports/figures/*.png'))
print(f'\n🖼  {len(figs)} Figures (reports/figures/):')
for f in figs:
    size = os.path.getsize(f) / 1024
    print(f'   {os.path.basename(f):45s} {size:6.0f} KB')

# Metrics
metrics = sorted(glob.glob('reports/metrics/*.json'))
print(f'\n📊 {len(metrics)} Metrics (reports/metrics/):')
for m in metrics:
    print(f'   {os.path.basename(m)}')
    with open(m) as f:
        data = json.load(f)
    if isinstance(data, dict):
        for k, v in list(data.items())[:5]:
            if isinstance(v, (int, float)):
                print(f'      {k}: {v}')

# Models
models_files = sorted(glob.glob('models/**/*.*', recursive=True))
print(f'\n🤖 {len(models_files)} Models:')
for m in models_files:
    size = os.path.getsize(m) / 1024
    print(f'   {m:50s} {size:8.0f} KB')

# Data files
data_files = sorted(glob.glob('data/**/*.csv', recursive=True) + glob.glob('data/**/*.json', recursive=True))
print(f'\n📦 {len(data_files)} Data Files:')
for d in data_files:
    size = os.path.getsize(d) / 1024
    print(f'   {d:50s} {size:8.0f} KB')

print(f'\n✅ Total files: {len(figs) + len(metrics) + len(models_files) + len(data_files)}')

## Cell 5: Download All Results as ZIP

In [ ]:
# Zip all outputs for download
import shutil

# Create ZIP of all results
shutil.make_archive('ragatherapy_results', 'zip', '.', 'reports')
shutil.make_archive('ragatherapy_models', 'zip', '.', 'models')
shutil.make_archive('ragatherapy_data', 'zip', '.', 'data')

print('📦 ZIP files created:')
for z in ['ragatherapy_results.zip', 'ragatherapy_models.zip', 'ragatherapy_data.zip']:
    if os.path.exists(z):
        size = os.path.getsize(z) / (1024 * 1024)
        print(f'   {z} ({size:.1f} MB)')

# Auto-download in Colab
try:
    from google.colab import files
    print('\n📥 Downloading ZIP files...')
    files.download('ragatherapy_results.zip')
    files.download('ragatherapy_models.zip')
    files.download('ragatherapy_data.zip')
except:
    print('\n💡 Not in Colab — ZIP files saved to current directory.')
    print('   Copy them manually from the file browser.')

## Cell 6: Quick Test — Try Your Own Symptoms

In [ ]:
# Interactive test — change the text below!
test_text = "I have anxiety and can't sleep at night"  # ← CHANGE THIS

print(f'\n📝 Input: "{test_text}"')
print('=' * 60)

# NLP Detection
features = engine.extract_features(test_text)
print(f'\n🔍 Detected Conditions: {features["detected_conditions"]}')
print(f'   Stress: {features.get("stress_score", 0):.2f}')
print(f'   Sleep Disruption: {features.get("sleep_disruption", 0):.2f}')
print(f'   Mood: {features.get("mood_score", 0):.2f}')

# Recommendation
result = raga_engine.recommend(test_text)
raga = result['raga_details']

print(f'\n🎵 Recommended Raga: {result["recommended_raga"]}')
print(f'   Confidence: {result["confidence"]}%')
print(f'   Emotion: {raga.get("primary_emotion", "N/A")}')
print(f'   Time: {raga.get("time_of_day", "N/A")}')
print(f'   Thaat: {raga.get("thaat", "N/A")}')

print(f'\n   Supporting Factors:')
for f in result['supporting_factors']:
    print(f'     ✓ {f}')

print(f'\n   Alternative Ragas:')
for i, alt in enumerate(result['alternative_ragas'], 1):
    print(f'     {i}. {alt["name"]} ({alt["score"]}%)')

print(f'\n   Description: {raga.get("brief_description", "N/A")}')